# AT1 VWAP Tracker v4

**v4 critical fix:** Self-tracked VWAP was double-counting fills.  
`record_fill()` incremented `shares_bought`, then `tracker.shares_bought = position` overwrote it,  
causing `total_cost` to inflate while `shares_bought` stayed correct → VWAP read $50 instead of $25.

**Fix:** Remove `record_fill()`. Directly accumulate `total_cost += delta * fill_price` and sync  
`shares_bought = position` from the API. No more double-counting.

**Same pacing from v3:** safety_buffer=30, depth_cap=60%, catchup=1.3, ahead_throttle=0.70.

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time
import base64
import tkinter as tk
import statistics
from collections import deque

In [24]:
class ApiException(Exception):
    pass

class RITGateway:
    def __init__(self, username, password, host='http://141.211.79.101:10002', use_http=True):
        protocol = 'http' if use_http else 'https'
        self.base_url = f'{protocol}://{host.split("://")[-1]}/v1'
        self.session = requests.Session()
        auth_value = base64.b64encode(f'{username}:{password}'.encode()).decode()
        self.session.headers.update({'Authorization': f'Basic {auth_value}'})

    def _request(self, method, endpoint, params=None):
        url = f'{self.base_url}/{endpoint}'
        while True:
            resp = getattr(self.session, method.lower())(url, params=params)
            if resp.status_code == 401: raise ApiException('Auth failed.')
            if resp.status_code == 429:
                time.sleep(float(resp.headers.get('Retry-After', 0.5))); continue
            if not resp.ok: raise ApiException(f'{resp.status_code} {resp.text}')
            return resp.json() if resp.text.strip() else None

    def get_case(self): return self._request('GET', 'case')
    def get_securities(self): return self._request('GET', 'securities')
    def get_security(self, ticker):
        for sec in self.get_securities():
            if sec.get('ticker') == ticker: return sec
        return None
    def get_book(self, ticker, limit=200):
        return self._request('GET', 'securities/book', params={'ticker': ticker, 'limit': limit})
    def submit_market_buy(self, ticker, quantity):
        return self._request('POST', 'orders', params={
            'ticker': ticker, 'type': 'MARKET', 'quantity': int(quantity), 'action': 'BUY'
        })

In [25]:
def get_historical_volume_profile():
    intervals = [
        ('9:30',780000),('9:40',520000),('9:50',400000),('10:00',350000),
        ('10:10',330000),('10:20',370000),('10:30',520000),('10:40',350000),
        ('10:50',290000),('11:00',280000),('11:10',300000),('11:20',310000),
        ('11:30',280000),('11:40',250000),('11:50',260000),('12:00',230000),
        ('12:10',220000),('12:20',230000),('12:30',250000),('12:40',240000),
        ('13:00',260000),('13:10',270000),('13:20',280000),('13:30',290000),
        ('13:40',300000),('13:50',310000),('14:00',320000),('14:10',330000),
        ('14:20',340000),('14:30',360000),('14:40',380000),('14:50',400000),
        ('15:00',450000),('15:10',500000),('15:20',600000),('15:30',700000),
        ('15:40',900000),('15:50',950000),
    ]
    volumes = np.array([v for _, v in intervals], dtype=float)
    return volumes / volumes.sum()

def build_tick_schedule(total_shares, total_ticks, volume_weights):
    n = len(volume_weights)
    tw = np.zeros(total_ticks)
    for t in range(total_ticks):
        tw[t] = volume_weights[min(int(t / total_ticks * n), n - 1)]
    tw /= tw.sum()
    spt = tw * total_shares
    cum = np.cumsum(spt)
    cum[-1] = total_shares
    return spt, cum

In [26]:
class AdaptiveTracker:
    """
    v4: VWAP tracking fixed.
    - total_cost accumulated externally via direct += delta * price
    - shares_bought synced from API position each tick
    - No record_fill method (removed to prevent double-count)
    """
    def __init__(self, total_shares=100000, total_ticks=300,
                 volume_weights=None, safety_buffer_ticks=30,
                 min_clip=100, max_clip=6000,
                 catchup_aggression=1.3, depth_fraction=0.60,
                 ahead_throttle=0.70):
        self.total_shares = total_shares
        self.total_ticks = total_ticks
        self.safety_buffer_ticks = safety_buffer_ticks
        self.min_clip = min_clip
        self.max_clip = max_clip
        self.catchup_aggression = catchup_aggression
        self.depth_fraction = depth_fraction
        self.ahead_throttle = ahead_throttle

        if volume_weights is None:
            volume_weights = get_historical_volume_profile()
        self.shares_per_tick, self.cumulative_target = build_tick_schedule(
            total_shares, total_ticks, volume_weights
        )

        # State — both set externally from main loop
        self.shares_bought = 0
        self.total_cost = 0.0

    def target_at_tick(self, tick):
        if tick < 0: return 0
        if tick >= self.total_ticks: return self.total_shares
        return self.cumulative_target[min(tick, self.total_ticks - 1)]

    def schedule_gap(self, tick):
        return self.target_at_tick(tick) - self.shares_bought

    def our_vwap(self):
        if self.shares_bought <= 0: return 0.0
        return self.total_cost / self.shares_bought

    def compute_buy_quantity(self, tick, ask_depth=0):
        remaining = max(0, self.total_shares - self.shares_bought)
        if remaining <= 0: return 0

        ticks_left = max(1, self.total_ticks - tick)
        base_qty = self.shares_per_tick[min(tick, self.total_ticks - 1)]
        gap = self.schedule_gap(tick)

        if gap > 500:
            effective_left = max(1, ticks_left - self.safety_buffer_ticks)
            base_qty += (gap / effective_left) * self.catchup_aggression
        elif gap < -2000:
            base_qty = base_qty * self.ahead_throttle * 0.5
        elif gap < -500:
            base_qty = base_qty * self.ahead_throttle

        # End-game
        if ticks_left <= self.safety_buffer_ticks:
            base_qty = max(base_qty, remaining / max(1, ticks_left) * 1.2)
        if ticks_left <= 10:
            base_qty = max(base_qty, remaining / max(1, ticks_left) * 1.5)
        if ticks_left <= 3:
            base_qty = remaining

        # Depth cap
        if ask_depth > 0:
            base_qty = min(base_qty, max(self.min_clip, ask_depth * self.depth_fraction))

        # Floor for late game
        if remaining > 0 and ticks_left <= 50 and base_qty < self.min_clip:
            base_qty = self.min_clip

        qty = max(0, min(remaining, base_qty))
        qty = min(qty, self.max_clip)
        qty = int(round(qty / 100)) * 100
        if qty < 100:
            if remaining <= 200 and ticks_left <= self.safety_buffer_ticks:
                qty = remaining
            else:
                return 0
        return min(qty, remaining)

    def get_status(self, tick):
        target = self.target_at_tick(tick)
        gap = self.schedule_gap(tick)
        remaining = max(0, self.total_shares - self.shares_bought)
        ticks_left = max(0, self.total_ticks - tick)

        if remaining <= 0: phase = 'COMPLETE'
        elif ticks_left <= 5: phase = 'FINAL_PUSH'
        elif ticks_left <= self.safety_buffer_ticks: phase = 'END_GAME'
        elif gap > 2000: phase = 'BEHIND'
        elif gap < -2000: phase = 'AHEAD'
        else: phase = 'ON_TRACK'

        return {
            'tick': tick, 'shares_bought': self.shares_bought,
            'target': int(target), 'gap': int(gap),
            'remaining': remaining, 'ticks_left': ticks_left,
            'phase': phase, 'our_vwap': self.our_vwap(),
            'pct_complete': self.shares_bought / self.total_shares * 100,
        }

In [27]:
class AT1Dashboard:
    _existing_root = None
    def __init__(self):
        if AT1Dashboard._existing_root is not None:
            try: AT1Dashboard._existing_root.destroy()
            except: pass
            AT1Dashboard._existing_root = None
        self.root = tk.Tk()
        AT1Dashboard._existing_root = self.root
        self.root.title('AT1 v4')
        self.root.geometry('700x620')
        self.root.configure(bg='#1a1a2e')
        self.stop_requested = False
        tk.Label(self.root, text='AT1 VWAP v4', font=('Helvetica', 20, 'bold'),
                 fg='#e94560', bg='#1a1a2e').pack(pady=8)
        self.shares_label = self._label('Shares: 0 / 100,000')
        self.pct_label = self._label('Progress: 0.0%')
        self.gap_label = self._label('Schedule Gap: 0')
        self.phase_label = self._label('Phase: WAITING')
        self.price_label = self._label('Last Price: $0.00')
        self.our_vwap_label = self._label('Our VWAP: --')
        self.mkt_vwap_label = self._label('Market VWAP: --')
        self.vwap_diff_label = self._label('VWAP Diff: --')
        self.last_order_label = self._label('Last Order: None')
        self.tick_label = self._label('Tick: 0')
        self.alert_label = tk.Label(self.root, text='WAITING', font=('Helvetica', 16, 'bold'),
                                     fg='white', bg='#16213e', width=35, height=2)
        self.alert_label.pack(pady=10)
        btn = tk.Frame(self.root, bg='#1a1a2e'); btn.pack(pady=8)
        tk.Button(btn, text='STOP', command=self.request_stop,
                  font=('Helvetica', 12, 'bold'), fg='white', bg='#e94560', width=14).pack()
        self.root.protocol('WM_DELETE_WINDOW', self.on_close)

    def _label(self, text):
        l = tk.Label(self.root, text=text, font=('Consolas', 12), fg='#eee', bg='#1a1a2e')
        l.pack(pady=2); return l
    def request_stop(self): self.stop_requested = True
    def on_close(self):
        self.stop_requested = True
        try: self.root.destroy()
        except: pass
        AT1Dashboard._existing_root = None
    def refresh(self):
        try: self.root.update_idletasks(); self.root.update()
        except: pass

    def update(self, status, last_price=0, market_vwap=0, last_order_qty=0, vwap_diff=0):
        self.shares_label.config(text=f"Shares: {status['shares_bought']:,} / 100,000")
        self.pct_label.config(text=f"Progress: {status['pct_complete']:.1f}%")
        self.gap_label.config(text=f"Schedule Gap: {status['gap']:+,}")
        self.phase_label.config(text=f"Phase: {status['phase']}")
        self.price_label.config(text=f"Last Price: ${last_price:.2f}")
        self.our_vwap_label.config(text=f"Our VWAP: ${status['our_vwap']:.4f}")
        self.mkt_vwap_label.config(text=f"Market VWAP: ${market_vwap:.4f}")
        self.vwap_diff_label.config(text=f"VWAP Diff: ${vwap_diff:+.4f}")
        self.last_order_label.config(text=f"Last Order: BUY {last_order_qty}")
        self.tick_label.config(text=f"Tick: {status['tick']} / {status['ticks_left']} left")

        ad = abs(vwap_diff)
        if status['remaining'] <= 0:
            t=f'DONE Gap=${vwap_diff:+.4f}'; bg='#0f3460' if ad<0.02 else '#e94560'
        elif status['our_vwap']<=0: t='STARTING...'; bg='#16213e'
        elif ad<0.02: t=f'EXCELLENT ${vwap_diff:+.4f}'; bg='#0f3460'
        elif ad<0.04: t=f'VERY GOOD ${vwap_diff:+.4f}'; bg='#16213e'
        elif ad<0.06: t=f'ON TRACK ${vwap_diff:+.4f}'; bg='#1a1a2e'
        elif ad<0.10: t=f'WATCH ${vwap_diff:+.4f}'; bg='#c77b30'
        else: t=f'DANGER ${vwap_diff:+.4f}'; bg='#e94560'
        self.alert_label.config(text=t, bg=bg)
        self.refresh()

In [28]:
def analyze_at1_run(log_df, save_prefix=None):
    print('=' * 70)
    print('              AT1 VWAP EXECUTION ANALYSIS v4')
    print('=' * 70)
    if log_df.empty: print('No data.'); return

    final = log_df.iloc[-1]
    print(f"\nTotal shares:   {final['shares_bought']:,}")
    print(f"Our VWAP:       ${final['our_vwap']:.4f}")
    print(f"Market VWAP:    ${final['market_vwap']:.4f}")
    print(f"VWAP Diff:      ${final['vwap_diff']:+.4f}")
    print(f"|VWAP Diff|:    ${abs(final['vwap_diff']):.4f}")
    print(f"Final tick:     {final['tick']}")

    ad = abs(final['vwap_diff'])
    for th, g in [(0.02,'VERY GOOD'),(0.04,'GOOD'),(0.06,'AVERAGE'),(0.10,'NEEDS IMPROVEMENT')]:
        if ad < th: grade=g; break
    else: grade='IMPROPER EXECUTION'
    print(f"Grade:          {grade}")

    if 'phase' in log_df.columns:
        print(f"\n--- PHASES ---")
        for p, s in log_df.groupby('phase')['order_qty'].sum().items():
            print(f"  {p}: {s:,.0f} shares")

    if 'gap' in log_df.columns:
        print(f"\n--- SCHEDULE ---")
        print(f"Max behind: {log_df['gap'].max():+,}")
        print(f"Max ahead:  {log_df['gap'].min():+,}")
        print(f"Avg gap:    {log_df['gap'].mean():+,.0f}")

    print(f"\n--- CONVERGENCE ---")
    v = log_df[log_df['our_vwap'] > 0]
    if not v.empty:
        for cp in [0.25,0.50,0.75,1.0]:
            idx = min(int(cp*len(v))-1, len(v)-1)
            if idx>=0:
                r=v.iloc[idx]
                print(f"  {cp*100:.0f}%: our=${r['our_vwap']:.4f} mkt=${r['market_vwap']:.4f} diff=${r['vwap_diff']:+.4f}")

    fig, axes = plt.subplots(3, 2, figsize=(15, 13))
    fig.suptitle('AT1 VWAP v4 Analysis', fontsize=16, fontweight='bold')

    ax=axes[0,0]
    ax.plot(log_df['tick'], log_df['shares_bought'], 'b-', lw=1.5, label='Actual')
    ax.plot(log_df['tick'], log_df['target'], 'r--', lw=1, alpha=0.7, label='Schedule')
    ax.fill_between(log_df['tick'], log_df['shares_bought'], log_df['target'], alpha=0.15, color='red')
    ax.set_title('Shares vs Schedule'); ax.legend(); ax.grid(True, alpha=0.3)

    ax=axes[0,1]
    v=log_df[log_df['our_vwap']>0]
    if not v.empty:
        ax.plot(v['tick'], v['our_vwap'], 'b-', lw=1.5, label='Our VWAP')
        ax.plot(v['tick'], v['market_vwap'], 'r-', lw=1.5, label='Market VWAP')
    ax.set_title('VWAP Convergence'); ax.legend(); ax.grid(True, alpha=0.3)

    ax=axes[1,0]
    ax.plot(log_df['tick'], log_df['gap'], lw=0.8)
    ax.axhline(y=0, color='green', ls='-', alpha=0.5)
    ax.fill_between(log_df['tick'], log_df['gap'], 0, alpha=0.2, where=log_df['gap']>0, color='red')
    ax.fill_between(log_df['tick'], log_df['gap'], 0, alpha=0.2, where=log_df['gap']<0, color='blue')
    ax.set_title('Schedule Gap'); ax.grid(True, alpha=0.3)

    ax=axes[1,1]
    ax.plot(log_df['tick'], log_df['mid'], lw=0.8)
    ax.set_title('TNX Mid Price'); ax.grid(True, alpha=0.3)

    ax=axes[2,0]
    f2=log_df[log_df['order_qty']>0]
    if not f2.empty: ax.bar(f2['tick'], f2['order_qty'], width=1, alpha=0.6, color='steelblue')
    ax.set_title('Order Sizes'); ax.grid(True, alpha=0.3)

    ax=axes[2,1]
    v=log_df[log_df['our_vwap']>0]
    if not v.empty:
        ax.plot(v['tick'], v['vwap_diff'], lw=0.8)
        ax.axhline(y=0, color='green', ls='-', alpha=0.5)
        ax.axhspan(-0.02, 0.02, alpha=0.1, color='green', label='Very Good')
        ax.axhspan(-0.04, 0.04, alpha=0.05, color='blue', label='Good')
    ax.set_title('VWAP Diff Over Time'); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_prefix:
        fig.savefig(f'{save_prefix}_analysis.png', dpi=150, bbox_inches='tight')
        print(f"\nChart saved: {save_prefix}_analysis.png")
    plt.show(); plt.close(fig)

In [29]:
def run_at1(gateway, tracker, ui, ticker='TNX', poll_delay=0.25, debug_every=5):
    """
    v4 VWAP fix: No more record_fill().
    We accumulate total_cost directly: total_cost += delta * last_price
    We sync shares_bought = position from API.
    These two are independent — no double-counting.
    """
    log = []
    prev_position = 0
    prev_tick = -1

    print(f'[START] AT1 v4 for {ticker}')
    print(f'[INFO] {tracker.total_ticks} ticks, {tracker.total_shares:,} target')

    while True:
        try:
            if ui.stop_requested: print('[STOP]'); break

            case = gateway.get_case()
            tick = case.get('tick', 0)
            if case.get('status') != 'ACTIVE':
                print(f'[END] tick={tick}'); break

            sec = gateway.get_security(ticker)
            if sec is None: time.sleep(poll_delay); continue

            position = int(float(sec.get('position', 0)))
            market_vwap = float(sec.get('vwap', 0))
            last_price = float(sec.get('last', 0))

            book = gateway.get_book(ticker)
            asks = book.get('asks', [])
            ask_depth = sum(l.get('quantity',0) - l.get('quantity_filled',0) for l in asks[:5])
            best_ask = float(asks[0]['price']) if asks else last_price
            bids = book.get('bids', [])
            best_bid = float(bids[0]['price']) if bids else last_price
            mid = (best_bid + best_ask) / 2 if best_bid and best_ask else last_price

            # ============================================
            # v4 FIX: Clean VWAP tracking
            # ============================================
            delta = position - prev_position
            if delta > 0:
                # Estimate fill price: use last_price (most recent trade)
                fill_price = last_price if last_price > 0 else mid
                tracker.total_cost += delta * fill_price

            # Sync position from API (single source of truth)
            tracker.shares_bought = position
            # ============================================

            # Tick-gated ordering
            order_qty = 0
            remaining = max(0, tracker.total_shares - position)

            if tick != prev_tick and remaining >= 100:
                buy_qty = tracker.compute_buy_quantity(tick, ask_depth=ask_depth)
                buy_qty = min(buy_qty, remaining)
                if buy_qty >= 100:
                    try:
                        gateway.submit_market_buy(ticker, buy_qty)
                        order_qty = buy_qty
                    except Exception as e:
                        print(f'[WARN] T{tick} order fail: {e}')

            our_vwap = tracker.our_vwap()
            vwap_diff = (our_vwap - market_vwap) if (our_vwap > 0 and market_vwap > 0) else 0.0
            st = tracker.get_status(tick)

            ui.update(st, last_price=last_price, market_vwap=market_vwap,
                      last_order_qty=order_qty, vwap_diff=vwap_diff)

            log.append({
                'tick': tick, 'shares_bought': position,
                'target': int(tracker.target_at_tick(tick)),
                'gap': int(tracker.target_at_tick(tick)) - position,
                'remaining': remaining, 'order_qty': order_qty,
                'last_price': last_price, 'best_bid': best_bid,
                'best_ask': best_ask, 'mid': mid, 'ask_depth': ask_depth,
                'our_vwap': our_vwap, 'market_vwap': market_vwap,
                'vwap_diff': vwap_diff,
                'market_volume': float(sec.get('volume', 0)),
                'phase': st['phase'],
            })

            if tick % debug_every == 0 and tick != prev_tick:
                print(
                    f'[T{tick:3d}] pos={position:6,} gap={int(tracker.target_at_tick(tick))-position:+5,} '
                    f'buy={order_qty:4,} ourV={our_vwap:.3f} mktV={market_vwap:.3f} '
                    f'diff={vwap_diff:+.4f} {st["phase"]}'
                )

            prev_position = position
            prev_tick = tick
            time.sleep(poll_delay)

        except Exception as e:
            print(f'[ERR]: {e}')
            import traceback; traceback.print_exc()
            time.sleep(poll_delay)

    return pd.DataFrame(log)

In [30]:
USERNAME = 'user4'
PASSWORD = 'user4'
HOST = 'http://141.211.79.101:10002'
TICKER = 'TNX'

gateway = RITGateway(USERNAME, PASSWORD, host=HOST)

try:
    case = gateway.get_case()
    total_ticks = case.get('ticks_per_period', 300)
    print(f'[INIT] {total_ticks} ticks')
except Exception as e:
    print(f'[INIT] default 300: {e}')
    total_ticks = 300

tracker = AdaptiveTracker(
    total_shares=100000,
    total_ticks=total_ticks,
    volume_weights=get_historical_volume_profile(),
    safety_buffer_ticks=30,
    min_clip=100,
    max_clip=6000,
    catchup_aggression=1.3,
    depth_fraction=0.60,
    ahead_throttle=0.70,
)

[INIT] 300 ticks


In [31]:
ui = AT1Dashboard()
log_df = run_at1(gateway, tracker, ui, ticker=TICKER, poll_delay=0.25, debug_every=5)

[START] AT1 v4 for TNX
[INFO] 300 ticks, 100,000 target
[T  0] pos=     0 gap= +673 buy= 700 ourV=0.000 mktV=0.000 diff=+0.0000 ON_TRACK
[T  5] pos= 3,500 gap= +542 buy= 700 ourV=24.988 mktV=25.010 diff=-0.0220 ON_TRACK
[T 10] pos= 6,400 gap= +336 buy= 400 ourV=24.991 mktV=25.007 diff=-0.0152 ON_TRACK
[T 15] pos= 8,500 gap= +482 buy= 400 ourV=24.981 mktV=24.996 diff=-0.0152 ON_TRACK
[T 20] pos=10,100 gap= +609 buy= 300 ourV=24.979 mktV=24.995 diff=-0.0152 ON_TRACK


In [10]:
analyze_at1_run(log_df, save_prefix='at1_v4')
log_df.to_csv('at1_v4_log.csv', index=False)
print('Saved at1_v4_log.csv')

NameError: name 'log_df' is not defined

In [11]:
import glob
files = sorted(glob.glob('at1_v*_log*.csv'))
if len(files) > 1:
    print('\n=== MULTI-ROUND ===')
    res = []
    for f in files:
        df = pd.read_csv(f); fin = df.iloc[-1]
        res.append({'file':f,'shares':fin['shares_bought'],'our':fin['our_vwap'],
                    'mkt':fin['market_vwap'],'diff':fin['vwap_diff'],'abs':abs(fin['vwap_diff'])})
    r = pd.DataFrame(res)
    print(r.to_string(index=False))
    print(f"\nAvg |diff|: ${r['abs'].mean():.4f}  Max: ${r['abs'].max():.4f}")
else:
    print('Run more rounds to compare.')


=== MULTI-ROUND ===
          file  shares      our      mkt     diff      abs
at1_v2_log.csv  100000 50.24846 25.09100 25.15746 25.15746
at1_v3_log.csv  100000 24.92024 24.92874 -0.00850  0.00850
at1_v4_log.csv  100000 25.10195 25.10735 -0.00540  0.00540

Avg |diff|: $8.3905  Max: $25.1575
